# Projekt starten mit jupyter lab

# Airbnb ML mit klassischen und modernen Machine-Learning-Algorithmen

In diesem Notebook werden die in der Projektarbeit beschriebenen Schritte 
praktisch umgesetzt: Datenaufbereitung, explorative Analyse, Modelltraining 
und Evaluation verschiedener Machine-Learning-Verfahren.

In [ ]:
"""
Initialisierung der Projektumgebung für das Airbnb-ML-Projekt.

Funktionen:
- Ermittelt das Projekt-Root anhand der Position von `requirements.txt`
- Setzt das Arbeitsverzeichnis (cwd) auf das Projekt-Root
- Fügt das Projekt-Root zu `sys.path` hinzu
- Installiert Abhängigkeiten aus `requirements.txt`
- Importiert zentrale Pipeline-Funktionen (Download, Preprocessing, Training)
"""

import sys
import os
from pathlib import Path

print("Starte Projekt-Umgebung.")

# Aktuelles Arbeitsverzeichnis (Startpunkt der Suche nach dem Projekt-Root)
current_work_dir = Path.cwd()

# Bestimmung des Projekt-Roots über typische Projektstrukturen

if (current_work_dir / "requirements.txt").exists():
    project_root = current_work_dir
elif (current_work_dir / "airbnb_ml" / "requirements.txt").exists():
    project_root = current_work_dir / "airbnb_ml"
elif (current_work_dir.parent / "requirements.txt").exists():
    project_root = current_work_dir.parent
else:
    raise RuntimeError(
        "Projekt-Root konnte nicht bestimmt werden."
        "Die Datei 'requirements.txt' wurde im aktuellen Verzeichnis,"
        "im Unterverzeichnis 'airbnb_ml' oder im Parent-Verzeichnis nicht gefunden."
    )

requirements_path = project_root / "requirements.txt"
print(f"📂 Projekt-Root: {project_root}")

# Wechsel des Arbeitsverzeichnisses ins Projekt-Root
os.chdir(project_root)
print(f"📍Arbeitsverzeichnis (cwd): {Path.cwd()}")

# Registrierung des Projekt-Roots im Modul-Suchpfad
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Installation bzw. Aktualisierung der Abhängigkeiten
if requirements_path.exists():
    print(f"📦 Installiere/überprüfe Abhängigkeiten aus {requirements_path.name}...")
    get_ipython().system(f'"{sys.executable}" -m pip install -r "{requirements_path}" -q')
    print("✅ Abhängigkeiten sind aktuell.")
else:
    print(f"⚠️ Warnung: Keine 'requirements.txt' unter {requirements_path} gefunden.")

# Import zentraler Bibliotheken und Projekt-Module
try:
    import colorama
    colorama.init(autoreset=True)
    from colorama import Fore, Style

    from data.download_data import main as run_download
    from preprocessing.data_preparation import load_and_prepare_data
    from scripts.model_training import main as run_training

    print(f"\n{Fore.GREEN}✅ Setup ist erfolgreich.{Style.RESET_ALL}")
    print("Alle Module wurden geladen und sind bereit für die Pipeline.")

except ImportError as e:
    print(f"\n❌ IMPORT FEHLER: {e}")
    print(f"   Fehlender Import: {e}")

In [ ]:
"""
run_pipeline.py

Zentrales Skript für die ML Pipeline.

Ausführung der folgenden Schritte:
1. Datenbeschaffung (Download)
2. Modelltraining (Training und Evaluation)
3. Reporting (Erstellung von Feature-Importance- und Ergebnis-Plots)

Das Skript stellt sicher, dass der Projekt-Root im Python-Pfad verfügbar ist,
um relative Importe der Module zu ermöglichen.
"""

import sys
import os
from pathlib import Path
import colorama
from colorama import Fore, Style

# Initialisierung für farbige Konsolenausgaben
colorama.init(autoreset=True)

# Config Pfad

try:
    # Versuch, den Pfad über __file__ zu ermitteln (Standard für .py Skripte)
    current_path = Path(__file__).resolve()
    project_root = current_path.parents[1]
except NameError:
    # Fallback für interaktive Umgebungen (z.B. Jupyter), wo __file__ nicht existiert.
    current_path = Path.cwd()

    # Prüfen, ob wir uns im Root oder einem Unterordner befinden
    if (current_path / "requirements.txt").exists():
        project_root = current_path
    else:
        # Unterordner (z.B. scripts/), daher eine Ebene hoch.
        project_root = current_path.parent

# Projekt-Root zum Systempfad hinzufügen, um Modul-Importe zu ermöglichen
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Imports

try:
    from data.download_data import main as run_download
    from scripts.model_training import main as run_training
    from scripts.plot_feature_importances import main as run_feature_plots
    from scripts.plot_model_results import main as run_results_plots

    print(f"Projekt-Umgebung initialisiert. Root: {project_root}")

except ImportError as e:
    print(f"{Fore.RED} Fehler beim Import der Module: {e}")
    sys.exit(1)


def run_full_pipeline():
    """
    Führt die gesamte ML-Pipeline Schritt für Schritt aus.
    Stoppt die Ausführung, wenn ein kritischer Schritt fehlschlägt.
    """

    # 1. Datenbeschaffung
    print(f"\n{Fore.CYAN} Start: Daten Pipeline {Style.RESET_ALL}")
    try:
        run_download()
    except Exception as e:
        print(f"{Fore.RED}Abbruch: Fehler beim Datendownload: {e}")
        return

    # 2. Modelltraining
    print(f"\n{Fore.CYAN} Start: Modelltraining{Style.RESET_ALL}")
    try:
        run_training()
    except Exception as e:
        print(f"{Fore.RED}Abbruch: Fehler beim Training: {e}")
        return

    # 3. Visualisierung & Reporting
    print(f"\n{Fore.CYAN} Start: Reporting & Visualisierung {Style.RESET_ALL}")
    try:
        print("Generiere Feature Importance Plots...")
        run_feature_plots()

        print("Generiere Ergebnis-Vergleiche...")
        run_results_plots()

    except Exception as e:
        print(f"{Fore.RED}Warnung: Fehler während der Visualisierungsphase: {e}")

    print(f"\n{Fore.GREEN}Pipeline erfolgreich beendet.{Style.RESET_ALL}")


if __name__ == "__main__":
    run_full_pipeline()